# 23-19 · Читаем и проверяем TOML-настройки

Практика к разделу [«Настройки проекта»](../../site/chapters/glava-23/23-22-nastrojki-proekta.html). Настоящий файл — `projects/python/safesort/src/safesort/config.py`.

## Цель

Разобрать `safesort.toml`-подобный текст через `tomllib` и построить из него `Config`-подобный объект — с теми же значениями по умолчанию, что и настоящий `load_config()`.

## Example

In [ ]:
import tomllib
from dataclasses import dataclass, field

DEFAULT_DESTINATION = "Sorted"
DEFAULT_EXCLUDE = (".git", ".venv")
DEFAULT_EXTENSIONS = {
    "documents": [".pdf", ".docx", ".txt", ".odt"],
    "images": [".jpg", ".jpeg", ".png", ".webp"],
}


@dataclass(frozen=True)
class Config:
    destination: str = DEFAULT_DESTINATION
    exclude: tuple = field(default_factory=lambda: DEFAULT_EXCLUDE)
    extensions: dict = field(default_factory=lambda: {k: list(v) for k, v in DEFAULT_EXTENSIONS.items()})


TEKST_TOML = """
destination = "Archive"
exclude = [".git", ".venv", "node_modules"]

[extensions]
documents = [".pdf", ".docx", ".txt"]
images = [".jpg", ".jpeg", ".png", ".webp"]
"""

raw = tomllib.loads(TEKST_TOML)
print(raw)

## Проверка результата — TOML разобран в обычный словарь Python

In [ ]:
assert raw["destination"] == "Archive"
assert raw["exclude"] == [".git", ".venv", "node_modules"]
assert raw["extensions"]["documents"] == [".pdf", ".docx", ".txt"]
print("Верно: tomllib.loads() вернул обычный словарь с ожидаемой структурой.")

## Строим Config из разобранного TOML

In [ ]:
def config_from_raw(raw):
    extensions = {k: list(v) for k, v in DEFAULT_EXTENSIONS.items()}
    extensions.update(raw.get("extensions", {}))
    return Config(
        destination=raw.get("destination", DEFAULT_DESTINATION),
        exclude=tuple(raw.get("exclude", list(DEFAULT_EXCLUDE))),
        extensions=extensions,
    )


nastrojki = config_from_raw(raw)
print(nastrojki)

## Starter

Заполните отмеченное место. Неизменённый starter не проходит tests.

In [ ]:
def config_iz_toml(text: str) -> Config:
    # TODO: parse text and apply defaults + category-specific overrides.
    raise NotImplementedError


nastrojki_minimalnye = config_iz_toml('destination = "Sorted2"\n')

## Task

Напишите `config_iz_toml(text)`. Новые категории должны добавляться поверх defaults, а названная встроенная категория должна заменяться.

## Tests

Запустите после task cell: есть основной пример и хотя бы один крайний случай.

In [ ]:
assert nastrojki_minimalnye.destination == "Sorted2"
assert nastrojki_minimalnye.extensions == DEFAULT_EXTENSIONS
with_books = config_iz_toml('[extensions]\nbooks = [".epub"]\n')
assert with_books.extensions["books"] == [".epub"]
assert with_books.extensions["documents"] == DEFAULT_EXTENSIONS["documents"]
overridden = config_iz_toml('[extensions]\ndocuments = [".md"]\n')
assert overridden.extensions["documents"] == [".md"]
print("Tests passed")

## Hint

Сначала скопируйте каждый список DEFAULT_EXTENSIONS, затем вызовите `.update()` с пользовательской таблицей.

## Solution

<details><summary>Показать решение после собственной попытки</summary>

```python
def config_iz_toml(text: str) -> Config:
    raw = tomllib.loads(text)
    extensions = {k: list(v) for k, v in DEFAULT_EXTENSIONS.items()}
    extensions.update(raw.get("extensions", {}))
    return Config(
        destination=raw.get("destination", DEFAULT_DESTINATION),
        exclude=tuple(raw.get("exclude", DEFAULT_EXCLUDE)),
        extensions=extensions,
    )


nastrojki_minimalnye = config_iz_toml('destination = "Sorted2"\n')
```

</details>